# Montandon EDA — schema, missingness, and linkage

Snapshot taken **2026-09-12** against `montandon-eoapi-stage.ifrc.org`. Every claim below sits directly above the cell that produces it. Raw pulls are cached under `data/raw/` (gitignored); the pull cell regenerates them.

**Caveat on all counts:** the staging database is being loaded live — `gdacs-events` grew from 43,595 to 43,723 items over ~4 hours on the day of the census. Treat every number as a snapshot.

## TL;DR

1. **The API schema (`api_schemas.py`) rejects ~97% of real items.** `monty:src_event_id` and `keywords` are required but present on only 1–31% of items. The schema matches what the ETL produces *today*; ~97% of the bank is backfilled history that predates those fields.
2. **The missing fields are missing by ETL generation, not by collection.** Items dated 2025+ have `keywords`; items dated 2026 also have `monty:src_event_id` and `processing:version 0.2.4`. Same pattern in every collection.
3. **`description` is mostly templated or placeholder text.** EM-DAT and GDACS are formulaic one-liners; 85% of IFRC event descriptions are the literal string `NA`, the rest raw HTML.
4. **`monty:corr_id` links events to impacts within a source (100%) but almost never across sources (0.8%), and by construction never will.** GDACS puts its own event ID where EM-DAT puts `1`. IFRC's own cookbook links across sources by date + bbox + hazard code instead.
5. **`monty:hazard_codes` mixes at least three vocabularies** in one field (EM-DAT, UNDRR-ISC, GLIDE). The ETL repo ships the crosswalk that normalises them.
6. **`keywords` is derived from `hazard_codes` + `country_codes` by the ETL**, not source text. Reconstructible for the 97% missing it; no extra signal for embeddings.
7. **EM-DAT geometry is full-resolution country polygons, up to 10 MB per item**, >99% of the payload, and duplicates `monty:country_codes`.

## Setup

In [1]:
from monty_tool.api_utils import get_pystac_client, get_collection_items
import os

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))


True

Smoke test through Jehan's validated path — works on the *newest* items, which is why the schema looked fine at first.

In [2]:
# Start small — max_items caps how many you pull
items = get_collection_items("emdat-events", max_items=20)

print(len(items))
first = items[0]
print(first.id, first.collection)
print(first.properties.title)
print(first.properties.monty_country_codes)
print(first.properties.start_datetime, first.properties.end_datetime)


20
emdat-event-2026-0571-ZWE emdat-events
Road in Zimbabwe
['ZWE']
2026-08-28 00:00:00+00:00 2026-08-28 00:00:00+00:00


## Collection census

The API doesn't report `numberMatched` on any endpoint, so `get_collection_counts()` pages every collection at 1,000 IDs per page (STAC `fields` extension strips everything else) and sums `numberReturned`. Hours on the staging server, so it's cached to `data/collection_counts.json`. `null` = uncounted: `usgs-events` passed 1M IDs after 6.5 h of paging and was cancelled; USGS hazards/impacts, PDC hazards/impacts, and DesInventar were never reached. All seven were already in the skip/sample tier.

Notable: every source has `events == hazards` exactly (1:1) except IFRC, which has **0** hazards. `ibtracs-hazards` is 722,837 — one item per storm-track observation, ~54 per storm. `cems-*`, `reference-events`, and `rsh-test2-events` are empty.

In [3]:
import json
from pathlib import Path

from monty_tool.api_utils import get_collection_counts

# data/ is gitignored, so this cache is local-only
counts_path = Path("../data/collection_counts.json")

if counts_path.exists():
    counts = json.loads(counts_path.read_text())
else:
    counts = get_collection_counts()
    counts_path.parent.mkdir(exist_ok=True)
    counts_path.write_text(json.dumps(counts, indent=2))

# keys starting with "_" are file metadata, not collections
rows = {k: v for k, v in counts.items() if not k.startswith("_")}
for cid, n in sorted(rows.items(), key=lambda kv: -(kv[1] or -1)):
    print(f"{n:>9,}  {cid}" if n is not None else f"{'(uncounted)':>9}  {cid}")
print(f"{sum(n for n in rows.values() if n):>9,}  TOTAL counted")

  722,837  ibtracs-hazards
  164,587  pdc-events
   64,637  emdat-impacts
   59,038  gdacs-hazards
   43,595  gdacs-events
   36,956  gdacs-impacts
   35,668  idmc-gidd-impacts
   32,558  idmc-idu-impacts
   25,308  emdat-events
   25,308  emdat-hazards
   21,161  idmc-idu-events
   19,230  idmc-gidd-events
   13,467  ibtracs-events
    8,523  glide-events
    8,523  glide-hazards
    2,199  ifrcevent-events
    1,826  gfd-impacts
    1,349  ifrcevent-impacts
      913  gfd-events
      913  gfd-hazards
        0  cems-events
        0  cems-hazards
        0  cems-impacts
        0  cems-response
(uncounted)  desinventar-events
(uncounted)  desinventar-impacts
        0  ifrcevent-hazards
(uncounted)  pdc-hazards
(uncounted)  pdc-impacts
        0  reference-events
        0  rsh-test2-events
(uncounted)  usgs-events
(uncounted)  usgs-hazards
(uncounted)  usgs-impacts
1,288,596  TOTAL counted


## Pulling the core collections

The core collections for all three tracks: EM-DAT (canonical disaster records + impacts, 1900–present), GDACS (second source with alert levels; the natural overlap test), and IFRC's own events/impacts (tiny, but the only operational-response data). ~137k items, 48 MB gzipped.

EM-DAT is pulled **without geometry**: its items carry full-resolution country MultiPolygons (mean 33 KB, max 10.8 MB for `emdat-event-2002-0495-CHL`), >99% of the payload, and a 1,000-item page times out the server's ~30 s gateway. `fields={"exclude": ["geometry"]}` is honoured server-side and cuts the pull from ~1.5 GB / 90 min to 10 MB / 4 min. `bbox` is kept; `country_codes` + any boundary set reproduces the polygon.

In [4]:
from pystac_client import Client

from monty_tool.api_utils import STAC_API_URL, _get_headers
from monty_tool.data_cache import load_collection, pull_collection, raw_cache_path

# EM-DAT geometry is >99% of its payload (full-res country polygons, up to 10 MB per Item)
# and duplicates monty:country_codes, so it's pulled without geometry.
TIER1 = {
    "emdat-events": False,
    "emdat-impacts": False,
    "gdacs-events": True,
    "ifrcevent-events": True,
    "ifrcevent-impacts": True,
}
CACHE_DIR = Path("../data/raw")

# The staging server silently drops connections under load; without a timeout a pull can hang forever
client = Client.open(STAC_API_URL, headers=_get_headers(), timeout=60)

for cid, geom in TIER1.items():
    if not raw_cache_path(cid, CACHE_DIR, geometry=geom).exists():
        pull_collection(cid, cache_dir=CACHE_DIR, client=client, geometry=geom)

raw = {cid: list(load_collection(cid, CACHE_DIR, geometry=geom)) for cid, geom in TIER1.items()}
for cid, items in raw.items():
    print(f"{len(items):>7,}  {cid}")

 25,308  emdat-events
 64,637  emdat-impacts
 43,723  gdacs-events
  2,199  ifrcevent-events
  1,349  ifrcevent-impacts


## Flatten to Polars

`items_to_frame` promotes `properties` to columns under their API names (`monty:corr_id`, not `monty_corr_id`) and flattens nested dicts with dotted keys (`monty:impact_detail.value`). Polars, to match the `embedding-setup` branch. Column null-rates *are* field coverage.

In [5]:
import polars as pl
from collections import Counter
from pydantic import ValidationError

from monty_tool.api_schemas import MontandonItem
from monty_tool.data_cache import items_to_frame

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(70)

frames = {cid: items_to_frame(items) for cid, items in raw.items()}
df = pl.concat(frames.values(), how="diagonal_relaxed")
df.shape

(137216, 30)

## 1. Field coverage

% of items with a non-null value, per field, per collection. **Every key the API returns is here** — including ones not in `api_schemas.py` (`monty:etl_id`, `monty:guid`, `processing:*`), which `extra="ignore"` drops silently.

What to look for:
- `keywords` 1–31%, `monty:src_event_id` 0–21% — both **required** in the schema.
- `monty:impact_detail.unit` is 100% on EM-DAT impacts and **0%** on IFRC impacts. Also required.
- `severitydata` is a GDACS field the schema puts on *impact* properties; here it's on 31% of GDACS *events*.
- `geometry_type` 0% on EM-DAT is our pull (`geometry=False`), not the API.

In [6]:
SKIP = {"id", "collection", "bbox", "n_links", "n_assets"}
coverage = (
    df.group_by("collection")
    .agg([(pl.col(c).is_not_null().mean() * 100).round(0).alias(c) for c in df.columns if c not in SKIP])
    .unpivot(index="collection", variable_name="field", value_name="pct")
    .pivot(on="collection", index="field", values="pct")
    .sort("field")
)
coverage

field,ifrcevent-impacts,emdat-events,gdacs-events,ifrcevent-events,emdat-impacts
str,f64,f64,f64,f64,f64
"""datetime""",100.0,100.0,100.0,100.0,100.0
"""description""",100.0,100.0,100.0,100.0,100.0
"""end_datetime""",100.0,100.0,100.0,100.0,100.0
"""geometry_type""",100.0,0.0,100.0,100.0,0.0
"""keywords""",2.0,3.0,31.0,1.0,3.0
"""monty:corr_id""",100.0,100.0,100.0,100.0,100.0
"""monty:country_codes""",100.0,100.0,100.0,100.0,100.0
"""monty:episode_number""",100.0,100.0,100.0,100.0,100.0
"""monty:etl_id""",100.0,100.0,100.0,100.0,100.0


## 2. ETL generations

`keywords` and `monty:src_event_id` aren't missing by *collection* — they're missing by *when the ETL ran*. Items dated 2025+ have `keywords`; items dated 2026 also have `monty:src_event_id` and `processing:version = 0.2.4` (`pystac-monty`). Everything older — ~97% of the bank — predates those fields. Same shape in every collection.

The ETL changelog (v1.0.0, 2026-06-30) describes "separate historical and latest pipelines": history was backfilled by an older code path. **Open question for IFRC:** will the historical pipeline be re-run on pystac-monty 0.2.4? If yes, the schema is right and we wait. If no, the schema needs `keywords` and `src_event_id` optional.

In [7]:
generations = (
    df.with_columns(pl.col("datetime").str.slice(0, 4).alias("year"))
    .group_by("collection", "year")
    .agg(
        pl.len().alias("n"),
        (pl.col("keywords").is_not_null().mean() * 100).round(0).alias("keywords_pct"),
        (pl.col("monty:src_event_id").is_not_null().mean() * 100).round(0).alias("src_event_id_pct"),
        (pl.col("processing:version").is_not_null().mean() * 100).round(0).alias("proc_version_pct"),
    )
    .filter(pl.col("year") >= "2022")
    .sort("collection", "year")
)
print("processing:version values:", df["processing:version"].drop_nulls().unique().to_list())
generations

processing:version values: ['0.2.4']


collection,year,n,keywords_pct,src_event_id_pct,proc_version_pct
str,str,u32,f64,f64,f64
"""emdat-events""","""2022""",564,0.0,0.0,0.0
"""emdat-events""","""2023""",578,0.0,0.0,0.0
"""emdat-events""","""2024""",542,0.0,0.0,0.0
"""emdat-events""","""2025""",449,92.0,0.0,0.0
"""emdat-events""","""2026""",241,100.0,100.0,100.0
"""emdat-impacts""","""2022""",1520,0.0,0.0,0.0
"""emdat-impacts""","""2023""",1499,0.0,0.0,0.0
"""emdat-impacts""","""2024""",1543,0.0,0.0,0.0
"""emdat-impacts""","""2025""",1247,92.0,0.0,0.0


## 3. Schema validation against real data

`MontandonItem.model_validate` on every cached item. EM-DAT gets a bbox-centre Point standing in for its excluded geometry, so this tests the schema, not our pull.

Pydantic tries all three `MontandonProperties` union members, so errors like `monty:hazard_detail missing` on an *event* are the hazard branch failing — noise. The root causes are the fields that fail on the *matching* branch: `monty:src_event_id` (99% EM-DAT/IFRC, 79% GDACS), `keywords` (97% / 69%), and `monty:impact_detail.unit` (99.7% of IFRC impacts).

The `embedding-setup` branch tightens `title`/`description` to non-blank (fine — 0 blanks found) and calls `properties.keywords` unconditionally in `extract_field_texts`, which will raise on ~97% of real items once validation passes.

Minimal schema change that validates the whole cache:
```python
keywords: list[str] = []
monty_src_event_id: str | None = Field(default=None, alias="monty:src_event_id")
# ImpactDetail
unit: str | None = None
```

In [8]:
def with_geometry(item: dict) -> dict:
    """Stand in a bbox-centre Point where geometry was excluded from the pull."""
    if item.get("geometry") is None and item.get("bbox"):
        b = item["bbox"]
        return {**item, "geometry": {"type": "Point", "coordinates": [(b[0] + b[2]) / 2, (b[1] + b[3]) / 2]}}
    return item

summary, reasons = [], []
for cid, items in raw.items():
    n = ok = 0
    errs: Counter[str] = Counter()
    for it in items:
        n += 1
        try:
            MontandonItem.model_validate(with_geometry(it))
            ok += 1
        except ValidationError as e:
            seen = set()
            for err in e.errors():
                loc = ".".join(str(x) for x in err["loc"] if not str(x).startswith("Montandon"))
                key = f"{loc} : {err['type']}"
                if key not in seen:
                    seen.add(key)
                    errs[key] += 1
    summary.append({"collection": cid, "n": n, "valid": ok, "invalid_pct": round(100 * (n - ok) / n, 1)})
    reasons += [{"collection": cid, "error": k, "items": v, "pct": round(100 * v / n, 1)} for k, v in errs.most_common()]

validation = pl.DataFrame(summary)
validation_reasons = pl.DataFrame(reasons).filter(pl.col("pct") >= 1).sort("collection", "items", descending=[False, True])
validation

collection,n,valid,invalid_pct
str,i64,i64,f64
"""emdat-events""",25308,241,99.0
"""emdat-impacts""",64637,661,99.0
"""gdacs-events""",43723,9307,78.7
"""ifrcevent-events""",2199,4,99.8
"""ifrcevent-impacts""",1349,4,99.7


In [9]:
validation_reasons

collection,error,items,pct
str,str,i64,f64
"""emdat-events""","""properties.monty:src_event_id : missing""",25067,99.0
"""emdat-events""","""properties.monty:hazard_detail : missing""",25067,99.0
"""emdat-events""","""properties.created : missing""",25067,99.0
"""emdat-events""","""properties.forecasted : missing""",25067,99.0
"""emdat-events""","""properties.severitydata : missing""",25067,99.0
"""emdat-events""","""properties.advisory_number : missing""",25067,99.0
"""emdat-events""","""properties.monty:impact_detail : missing""",25067,99.0
"""emdat-events""","""properties.keywords : missing""",24655,97.4
"""emdat-impacts""","""properties.monty:src_event_id : missing""",63976,99.0


## 4. Text fields

What's actually in `title` and `description`? This decides which collection is a usable NLP / embedding corpus.

- **EM-DAT:** `Flood in Lushoto, Korogwe districts (Tanga region), United Republic of Tanzania of February 1993` — formulaic `<Hazard> in <Place>, <Country> of <Month Year>`. The only content not already in `hazard_codes` / `country_codes` / `datetime` is the sub-national place name.
- **GDACS:** `Green M 4.6 Earthquake in Indonesia at: 28 Apr 2005 04:48:13.` — fully templated. Embeddings will cluster by template, not by event.
- **IFRC:** `NA` for 85% of events; the rest is raw HTML from a rich-text editor. The ~300 real ones are long (p90 1.3k chars, max 79k) and are the only free-form narrative among the five collections pulled. Needs HTML stripping before any NLP.

Implication for the embedding track: `keywords` is embeddable on <1% of items, `description` is templated on the two big collections. Title/description similarity is still useful as a *matching feature* across sources; it's weak as a clustering target.

In [10]:
text_stats = (
    df.with_columns(
        pl.col("description").str.len_chars().alias("desc_len"),
        pl.col("title").str.len_chars().alias("title_len"),
    )
    .group_by("collection")
    .agg(
        pl.len().alias("n"),
        pl.col("desc_len").median().alias("desc_median"),
        pl.col("desc_len").quantile(0.9).alias("desc_p90"),
        pl.col("desc_len").max().alias("desc_max"),
        pl.col("description").n_unique().alias("desc_unique"),
        (pl.col("description") == "NA").sum().alias("desc_is_NA"),
        pl.col("description").str.contains("<[a-z]+").sum().alias("desc_has_html"),
        pl.col("title_len").median().alias("title_median"),
    )
    .sort("collection")
)
text_stats

collection,n,desc_median,desc_p90,desc_max,desc_unique,desc_is_NA,desc_has_html,title_median
str,u32,f64,f64,u32,u32,u32,u32,f64
"""emdat-events""",25308,65.0,151.0,2909,25196,0,0,28.0
"""emdat-impacts""",64637,70.0,171.0,2124,24369,0,0,43.0
"""gdacs-events""",43723,63.0,78.0,343,30427,0,0,23.0
"""ifrcevent-events""",2199,2.0,1258.0,78932,326,1871,268,21.0
"""ifrcevent-impacts""",1349,503.0,4367.0,78932,218,641,613,38.0


In [11]:
for cid in raw:
    print(f"--- {cid} ---")
    for d in frames[cid].sample(4, seed=1)["description"].to_list():
        print("  ", d[:160].replace("\n", " "))

--- emdat-events ---
   Explosion (Miscellaneous) in Mexico city, Mexico of December 1988
   Earthquake in Nehbandan area, Iran (Islamic Republic of) of January 1993
   Flood in Castellón and Valencia Provinces (Valencian Community); Tarragona, Toledo, Spain of August 2021
   Road in Between Turin - Milan, Italy of February 1993
--- emdat-impacts ---
   Drought in Regions VI, IX, X, XI and XII, Philippines of December 1990
   Storm in Dinant, Liege regions, Belgium of January 1995
   Flood in East Aceh and Aceh Tamiang (Aceh Province, northern Sumatra), Indonesia of December 2021
   Earthquake in Paphos, Nicosia areas, Cyprus of February 1995
--- gdacs-events ---
   Green M 4.8 Earthquake in Indonesia at: 28 Apr 2005 09:36:43.
   Green M 5.2 Earthquake in Iran at: 30 Mar 2006 19:36:17.
   Green Tropical Cyclone LALA-26 in United States from: 12 Aug 2026 to: 28 Aug 2026 .
   Green M 4.3 Earthquake in Brazil at: 03 Apr 2006 03:33:47.
--- ifrcevent-events ---
   NA
   NA
   <p class="MsoN

In [12]:
# The IFRC "description" field is a placeholder 85% of the time
frames["ifrcevent-events"].group_by("description").len().sort("len", descending=True).head(5)

description,len
str,u32
"""NA""",1871
"""<br data-mce-bogus=""1"">""",4
"""<p>Five days of cumulative heavy rains were witness in Angola, resulti…",1
"""<p class=""MsoNormal"" style=""margin: 0cm; font-size: 12pt; font-family:…",1
"""<p class=""MsoNormal"" style=""text-align: justify;"">Multiple Fires have …",1


## 5. `monty:corr_id` linkage

`corr_id` is the documented cross-source correlation key. How well does it actually link?

- **Within-source:** every EM-DAT impact and every IFRC impact joins to an event on `corr_id` (100%).
- **Cross-source:** only 470 of 56,712 event `corr_id`s (0.8%) appear in more than one source collection.

In [13]:
corr = df.group_by("monty:corr_id").agg(
    pl.col("collection").unique().sort().alias("collections"),
    pl.len().alias("n_items"),
)
linkage = corr.group_by("collections").agg(
    pl.len().alias("n_corr_ids"),
    pl.col("n_items").sum().alias("n_items"),
).sort("n_corr_ids", descending=True)

events = df.filter(pl.col("collection").str.ends_with("-events"))
n_sources = events.group_by("monty:corr_id").agg(pl.col("collection").n_unique().alias("n_sources"))
multi = n_sources.filter(pl.col("n_sources") > 1).height
print(f"unique corr_ids: {corr.height:,}")
print(f"event corr_ids seen in >1 source: {multi:,} of {n_sources.height:,} ({100 * multi / n_sources.height:.1f}%)")
for src in ["emdat", "ifrcevent"]:
    ev, im = frames[f"{src}-events"], frames[f"{src}-impacts"]
    joined = im.join(ev.select("monty:corr_id").unique(), on="monty:corr_id", how="semi").height
    print(f"{src}-impacts -> {src}-events join: {100 * joined / im.height:.1f}%")
linkage

unique corr_ids: 56,715
event corr_ids seen in >1 source: 470 of 56,712 (0.8%)
emdat-impacts -> emdat-events join: 100.0%
ifrcevent-impacts -> ifrcevent-events join: 100.0%


collections,n_corr_ids,n_items
list[str],u32,u32
"[""gdacs-events""]",29507,43203
"[""emdat-events"", ""emdat-impacts""]",23855,87100
"[""ifrcevent-events""]",1723,1724
"[""emdat-events""]",828,830
"[""ifrcevent-events"", ""ifrcevent-impacts""]",329,1426
"[""emdat-events"", ""emdat-impacts"", ""gdacs-events""]",327,1915
"[""emdat-events"", ""emdat-impacts"", ""ifrcevent-events""]",50,306
"[""emdat-events"", ""emdat-impacts"", … ""ifrcevent-impacts""]",48,435
"[""gdacs-events"", ""ifrcevent-events"", ""ifrcevent-impacts""]",21,105


### Why it can't work across sources

From the ETL repo (`pystac_monty/paring.py`), `corr_id` is deterministic:
`{YYYYMMDD}-{country}-{geoblock}-{hazard_cluster}-{episode}-GCDB`. Three format generations coexist in the bank, and **GDACS puts its own event ID in the `episode` slot** (`…-GH0101-1732529-GCDB`) where EM-DAT and IFRC put `1`. So a GDACS and an EM-DAT record of the same event can never share a `corr_id`, in any format. The geoblock in the current format makes matching stricter still — the current-format IDs have a **0.0%** cross-source rate.

IFRC's own cookbook (recipe 07, Türkiye–Syria 2023) doesn't use `corr_id` across sources either: it links by `bbox` + `datetime` window + a hand-enumerated list of hazard codes across every vocabulary. That's the approach to replicate, with the crosswalk replacing the hand-typed lists.

In [14]:
# corr_id format generations. Current format has a numeric geoblock between country and cluster.
CURRENT = r"^\d{8}-[A-Z]{3}-\d+-[A-Z]{2}\d{4}-\d+-GCDB$"
ev_fmt = (
    df.filter(pl.col("collection").str.ends_with("-events"))
    .with_columns(
        pl.col("datetime").str.slice(0, 4).alias("year"),
        pl.col("monty:corr_id").str.contains(CURRENT).alias("current_format"),
    )
)
print(ev_fmt.group_by("collection", "current_format").agg(pl.len().alias("n"), pl.col("monty:corr_id").first().alias("example")).sort("collection", "current_format"))
print()
by_fmt = ev_fmt.group_by("monty:corr_id").agg(pl.col("collection").n_unique().alias("n_src"), pl.col("current_format").first())
by_fmt.group_by("current_format").agg(
    pl.len().alias("corr_ids"),
    (pl.col("n_src") > 1).sum().alias("multi_source"),
    ((pl.col("n_src") > 1).mean() * 100).round(2).alias("multi_source_pct"),
)

shape: (6, 4)
┌──────────────────┬────────────────┬───────┬─────────────────────────────────────────┐
│ collection       ┆ current_format ┆ n     ┆ example                                 │
│ ---              ┆ ---            ┆ ---   ┆ ---                                     │
│ str              ┆ bool           ┆ u32   ┆ str                                     │
╞══════════════════╪════════════════╪═══════╪═════════════════════════════════════════╡
│ emdat-events     ┆ false          ┆ 25067 ┆ 20251216-IDN-MH0600-1-GCDB              │
│ emdat-events     ┆ true           ┆ 241   ┆ 20260828-ZWE-638247-TL0405-1-GCDB       │
│ gdacs-events     ┆ false          ┆ 33651 ┆ 20260309-GTM-GH0101-1691930-GCDB        │
│ gdacs-events     ┆ true           ┆ 10072 ┆ 20260911-COL-848317-GH0101-1732529-GCDB │
│ ifrcevent-events ┆ false          ┆ 2194  ┆ 20260406-SLB-MH0306-1-GCDB              │
│ ifrcevent-events ┆ true           ┆ 5     ┆ 20260826-NPL-1065121-MH0600-1-GCDB      │
└─────────────────

current_format,corr_ids,multi_source,multi_source_pct
bool,u32,u32,f64
false,46415,470,1.01
true,10297,0,0.0


## 6. Categoricals and geometry

- **Hazard codes:** most items carry one code, but ~27k carry 3 or 5 — one per taxonomy. Top values mix EM-DAT (`nat-hyd-flo-riv`), UNDRR-ISC (`GH0001`, `EN0013`), and GLIDE (`FL`, `WF`) vocabularies in the same field. The ETL repo's `HazardProfiles.csv` (303 rows) crosswalks all of them to a `cluster_label` / `family_label`.
- **Countries:** 98% single-country.
- **Geometry:** GDACS = Point; IFRC = Polygon/MultiPolygon; EM-DAT = country MultiPolygons (excluded here).
- **Date ranges:** EM-DAT 1900–2026; GDACS 2000–2026; IFRC 1970–2026.

In [15]:
print("hazard codes per item:")
print(df["monty:hazard_codes"].list.len().value_counts().sort("monty:hazard_codes"))
print("\ntop hazard codes (note the mixed vocabularies — EM-DAT `nat-hyd-flo-riv`, UNDRR `GH0001`, GLIDE `FL`):")
df.explode("monty:hazard_codes").group_by("monty:hazard_codes").len().sort("len", descending=True).head(15)

hazard codes per item:
shape: (4, 2)
┌────────────────────┬────────┐
│ monty:hazard_codes ┆ count  │
│ ---                ┆ ---    │
│ u32                ┆ u32    │
╞════════════════════╪════════╡
│ 1                  ┆ 108948 │
│ 2                  ┆ 871    │
│ 3                  ┆ 16857  │
│ 5                  ┆ 10540  │
└────────────────────┴────────┘

top hazard codes (note the mixed vocabularies — EM-DAT `nat-hyd-flo-riv`, UNDRR `GH0001`, GLIDE `FL`):


/var/folders/0s/rn0t7r9s2jl33n8wchyp4f600000gn/T/ipykernel_34446/675812575.py:4: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  df.explode("monty:hazard_codes").group_by("monty:hazard_codes").len().sort("len", descending=True).head(15)


monty:hazard_codes,len
str,u32
"""EN0013""",13044
"""nat-cli-wil-wil""",11318
"""nat-met-sto-tro""",11274
"""WF""",10983
"""nat-hyd-flo-riv""",10708
"""EN0205""",10597
"""GH0004""",10446
"""GH0001""",10446
"""GH0003""",10446


In [16]:
print("countries per item:")
print(df["monty:country_codes"].list.len().value_counts().sort("monty:country_codes"))
print("\ntop countries:")
df.explode("monty:country_codes").group_by("monty:country_codes").len().sort("len", descending=True).head(10)

countries per item:
shape: (14, 2)
┌─────────────────────┬────────┐
│ monty:country_codes ┆ count  │
│ ---                 ┆ ---    │
│ u32                 ┆ u32    │
╞═════════════════════╪════════╡
│ 1                   ┆ 134482 │
│ 2                   ┆ 2358   │
│ 3                   ┆ 134    │
│ 4                   ┆ 80     │
│ 5                   ┆ 106    │
│ 6                   ┆ 11     │
│ 7                   ┆ 4      │
│ 8                   ┆ 18     │
│ 9                   ┆ 2      │
│ 10                  ┆ 1      │
│ 14                  ┆ 1      │
│ 15                  ┆ 1      │
│ 16                  ┆ 1      │
│ 26                  ┆ 17     │
└─────────────────────┴────────┘

top countries:


/var/folders/0s/rn0t7r9s2jl33n8wchyp4f600000gn/T/ipykernel_34446/1707959324.py:4: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  df.explode("monty:country_codes").group_by("monty:country_codes").len().sort("len", descending=True).head(10)


monty:country_codes,len
str,u32
"""CHN""",8214
"""USA""",7180
"""IND""",6358
"""IDN""",5692
"""RUS""",5136
"""PHL""",4895
"""AUS""",4848
"""BRA""",3305
"""JPN""",3105


In [17]:
print(df.group_by("collection", "geometry_type").len().sort("collection"))
print(df.group_by("collection").agg(
    pl.col("datetime").min().str.slice(0, 10).alias("min_date"),
    pl.col("datetime").max().str.slice(0, 10).alias("max_date"),
).sort("collection"))
frames["emdat-impacts"].group_by("monty:impact_detail.category", "monty:impact_detail.type").len().sort("len", descending=True)

shape: (7, 3)
┌───────────────────┬───────────────┬───────┐
│ collection        ┆ geometry_type ┆ len   │
│ ---               ┆ ---           ┆ ---   │
│ str               ┆ str           ┆ u32   │
╞═══════════════════╪═══════════════╪═══════╡
│ emdat-events      ┆ null          ┆ 25308 │
│ emdat-impacts     ┆ null          ┆ 64637 │
│ gdacs-events      ┆ Point         ┆ 43723 │
│ ifrcevent-events  ┆ Polygon       ┆ 1218  │
│ ifrcevent-events  ┆ MultiPolygon  ┆ 981   │
│ ifrcevent-impacts ┆ MultiPolygon  ┆ 684   │
│ ifrcevent-impacts ┆ Polygon       ┆ 665   │
└───────────────────┴───────────────┴───────┘
shape: (5, 3)
┌───────────────────┬────────────┬────────────┐
│ collection        ┆ min_date   ┆ max_date   │
│ ---               ┆ ---        ┆ ---        │
│ str               ┆ str        ┆ str        │
╞═══════════════════╪════════════╪════════════╡
│ emdat-events      ┆ 1900-01-01 ┆ 2026-08-28 │
│ emdat-impacts     ┆ 1900-01-01 ┆ 2026-08-28 │
│ gdacs-events      ┆ 2000-01-01 ┆ 202

monty:impact_detail.category,monty:impact_detail.type,len
str,str,u32
"""people""","""affected_total""",27284
"""people""","""death""",20198
"""people""","""injured""",8620
"""total_affected""","""cost""",5843
"""people""","""displaced_total""",2692


## What the upstream repos tell us

Two IFRC repos explain most of the above. Neither is worth installing as a dependency; both are worth borrowing from.

**[`IFRCGo/montandon-etl`](https://github.com/IFRCGo/montandon-etl)** (+ `pystac-monty` submodule) — the ETL that produces every item.
- `keywords = hazard_profiles.get_keywords(hazard_codes) + country_codes` in every source. Derived, not sourced.
- `paring.py` — the `corr_id` construction above.
- `HazardProfiles.csv` — the hazard-code crosswalk. `geo_blocks-0.2.parquet` — the spatial grid.
- `extension.py` — canonical field list and enums (`MontyRoles`, `MontyImpactType`, `MontyResponseType`, …) and a `monty:response_detail` object we haven't seen because no response collection has data yet. Schema URI `monty-stac-extension/v1.3.0`.

**[`IFRCGo/montandon-notebooks`](https://github.com/IFRCGo/montandon-notebooks)** ([cookbook](https://ifrcgo.org/montandon-notebooks/)) — nine recipes.
- Recipe 07 links across sources by date + bbox + hazard code, not `corr_id`.
- Recipe 08: the API supports **CQL2 filtering server-side** — `a_overlaps` on `monty:hazard_codes`, `a_contains` on `monty:country_codes`, `t_intersects` on `datetime`, comparisons on `monty:impact_detail.type/.value`. `/queryables` lists filterable fields. Targeted questions don't need a full pull. Filter by *collection*, not `roles` (server bug).
- Per-source field mapping docs at `monty-stac-extension/model/sources/<SOURCE>`, linked from each collection's `describedby`.

## API behaviour worth knowing

- No `numberMatched` anywhere; counting means paging IDs.
- ~30 s gateway timeout; the server drops connections under load with no error. `pystac_client.Client.open(timeout=…)` is the only thing that stops a hang — Jehan's `get_pystac_client()` doesn't set it.
- Max page size 10,000, but page size has to be per-collection: 1,000 is fine for GDACS (2 KB/item), 100 is the ceiling for EM-DAT with geometry.
- `fields={"exclude": ["geometry"]}` is honoured server-side.